Save data in parquet format

In [237]:
import pandas as pd

df = pd.read_csv("../data/raw/cfpb_raw_2026-02-28_narrative_only.csv")

df.to_parquet("../data/interim/cfpb_raw_2026-02-28_narrative_only.parquet", compression="snappy")

Load Data

In [238]:
df = pd.read_parquet("../data/interim/cfpb_raw_2026-02-28_narrative_only.parquet")

print(df.shape)          
print(df.columns)                 

(1404900, 18)
Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Consumer consent provided?',
       'Submitted via', 'Date sent to company', 'Company response to consumer',
       'Timely response?', 'Consumer disputed?', 'Complaint ID'],
      dtype='str')


Check number of unique complaint categories

In [243]:
print(df["Product"].nunique())

21


Check class balance

In [244]:
df["Product"].value_counts()

Product
Credit reporting or other personal consumer reports                             628016
Credit reporting, credit repair services, or other personal consumer reports    304342
Debt collection                                                                 152342
Checking or savings account                                                      63218
Mortgage                                                                         52591
Money transfer, virtual currency, or money service                               41927
Credit card or prepaid card                                                      41319
Credit card                                                                      41269
Student loan                                                                     22273
Vehicle loan or lease                                                            17960
Credit reporting                                                                 11762
Payday loan, title loan, or persona

Check very short narratives

In [245]:
df["n_words"] = df["Consumer complaint narrative"].str.split().str.len()

In [246]:
df["n_words"].describe()

count    1.404900e+06
mean     1.762166e+02
std      2.220826e+02
min      1.000000e+00
25%      6.000000e+01
50%      1.160000e+02
75%      2.110000e+02
max      6.469000e+03
Name: n_words, dtype: float64

In [247]:
sorted_df = df.sort_values("n_words")

In [248]:
sorted_df[["Consumer complaint narrative", "n_words"]].head(20)

,Consumer complaint narrative,n_words
721321,XX/XX/XXXX,1
912265,XX/XX/XXXX,1
151266,XX/XX/XXXX,1
260006,XX/XX/XXXXXX/XX/XXXX,1
618658,IhaveanaccountonmyfilethatisnotmineIfiledafrau...,1
1319478,TransunionisreportinganinacurratelistingofChap...,1
976945,XX/XX/XXXXXXXX,1
1187403,Letsstartthiscomplaintoffbysayingthatnotonlydo...,1
111603,XX/XX/XXXX,1
197068,CapitalOnerefusedtoforceXXXXltoreimbursethemfo...,1


In [249]:
(df["n_words"] < 5).sum()

np.int64(1204)

Remove very short narratives

In [250]:
df = df[df["n_words"] >= 5]

Detect complaints containing only masks

In [251]:
mask_only = df["Consumer complaint narrative"].str.fullmatch(r"[Xx/\s]+")
mask_only.sum()

np.int64(245)

Remove complaints containing only masks

In [252]:
df = df[~mask_only]

In [253]:
df.shape

(1403451, 19)

Check duplicated narratives

In [254]:
df["Consumer complaint narrative"].duplicated().sum()

np.int64(387551)

Remove duplicate narratives

In [255]:
df = df.drop_duplicates(subset=["Consumer complaint narrative"])

In [256]:
df.shape

(1015900, 19)

Final Department Mapping

In [257]:
product_to_department = {

    # Bank accounts
    "Bank account or service": "Bank accounts",
    "Checking or savings account": "Bank accounts",

    # Consumer loans
    "Consumer Loan": "Consumer loans",
    "Vehicle loan or lease": "Consumer loans",

    # Credit cards (includes prepaid)
    "Credit card": "Card services",
    "Credit card or prepaid card": "Card services",
    "Prepaid card": "Card services",

    # Credit reporting
    "Credit reporting": "Credit reporting",
    "Credit reporting or other personal consumer reports": "Credit reporting",
    "Credit reporting, credit repair services, or other personal consumer reports": "Credit reporting",
    "Debt or credit management": "Credit reporting",

    # Debt collection
    "Debt collection": "Debt collection",

    # Money transfer services
    "Money transfer, virtual currency, or money service": "Money transfer services",
    "Money transfers": "Money transfer services",
    "Virtual currency": "Money transfer services",
    "Other financial service": "Money transfer services",

    # Mortgage
    "Mortgage": "Mortgage",

    # Payday / personal loans
    "Payday loan": "Payday / personal loans",
    "Payday loan, title loan, or personal loan": "Payday / personal loans",
    "Payday loan, title loan, personal loan, or advance loan": "Payday / personal loans",

    # Student loan
    "Student loan": "Student loan"
}

df["Department"] = df["Product"].map(product_to_department)

Check final department distribution

In [333]:
df["Department"].value_counts(normalize = True)

Department
Credit reporting           0.582926
Debt collection            0.130451
Card services              0.080420
Bank accounts              0.066205
Mortgage                   0.051714
Money transfer services    0.033294
Student loan               0.021819
Consumer loans             0.020997
Payday / personal loans    0.012173
Name: proportion, dtype: float64

Score-based system to engineer priority target

In [271]:
import re

def assign_priority_credit_reporting(text):
    text = str(text).lower()
    score = 0

    # -------------------------
    # 1. CRITICAL (identity theft / fraud / major harm)
    # -------------------------
    fraud_signals = [
        "identity theft", "fraud", "fraudulent",
        "not mine", "unauthorized",
        "accounts opened", "stolen identity"
    ]
    score += sum(4 for kw in fraud_signals if kw in text)

    # -------------------------
    # 2. MAJOR CREDIT DAMAGE
    # -------------------------
    damage_signals = [
        "credit score", "denied", "loan denied",
        "hurting my credit", "affecting my score"
    ]
    score += sum(2 for kw in damage_signals if kw in text)

    # -------------------------
    # 3. INACCURATE / DUPLICATE REPORTING
    # -------------------------
    accuracy_signals = [
        "incorrect", "inaccurate",
        "duplicate", "wrong",
        "misreported", "error"
    ]
    score += sum(2 for kw in accuracy_signals if kw in text)

    # -------------------------
    # 4. DISPUTE FAILURE
    # -------------------------
    dispute_signals = [
        "dispute", "no response",
        "not investigated", "failed to investigate",
        "30 days", "no results",
        "still remains"
    ]
    score += sum(2 for kw in dispute_signals if kw in text)

    # -------------------------
    # 5. LEGAL / FCRA REFERENCES 
    # -------------------------
    legal_signals = [
        "fcra", "fair credit reporting",
        "section 609", "section 611",
        "violation", "law", "legal action"
    ]
    score += sum(2 for kw in legal_signals if kw in text)

    # -------------------------
    # 6. UNAUTHORIZED INQUIRIES
    # -------------------------
    inquiry_signals = [
        "inquiry", "hard pull",
        "unauthorized inquiry"
    ]
    score += sum(2 for kw in inquiry_signals if kw in text)

    # -------------------------
    # 7. LARGE AMOUNTS / MULTIPLE ACCOUNTS
    # -------------------------
    amounts = re.findall(r"\$[\d,]+", text)
    score += min(len(amounts), 4)

    # -------------------------
    # 8. COMPLEXITY (long structured disputes)
    # -------------------------
    word_count = len(text.split())
    if word_count > 250:
        score += 2
    elif word_count > 120:
        score += 1

    # -------------------------
    # 9. PERSISTENCE / LONG ISSUE
    # -------------------------
    persistence_signals = [
        "months", "years",
        "multiple times",
        "still", "ongoing"
    ]
    score += sum(1 for kw in persistence_signals if kw in text)

    # -------------------------
    # FINAL CLASSIFICATION
    # -------------------------
    if score >= 13:
        return "critical"
    elif score >= 6:
        return "high_priority"
    else:
        return "standard"

In [272]:
mask = df["Department"].str.lower() == "credit reporting"

df.loc[mask, "priority"] = df.loc[mask, "Consumer complaint narrative"].apply(assign_priority_credit_reporting)
df.loc[mask, "priority"].value_counts(normalize=True)

priority
standard         0.405095
high_priority    0.337612
critical         0.257294
Name: proportion, dtype: float64

In [274]:
def assign_priority_debt_collection(text):
    text = str(text).lower()
    score = 0

    # -------------------------
    # 1. CRITICAL (legal threats / lawsuits / identity theft)
    # -------------------------
    critical_signals = [
        "lawsuit", "court", "sued",
        "legal action", "summons",
        "wage garnishment", "arrest",
        "identity theft", "fraud",
        "not mine"
    ]
    score += sum(4 for kw in critical_signals if kw in text)

    # -------------------------
    # 2. HARASSMENT / ABUSE (VERY COMMON HERE)
    # -------------------------
    harassment_signals = [
        "harass", "harassment",
        "calling repeatedly",
        "multiple calls", "spam",
        "threatening", "abusive",
        "calling my family", "calling work"
    ]
    score += sum(3 for kw in harassment_signals if kw in text)

    # -------------------------
    # 3. INVALID / DISPUTED DEBT
    # -------------------------
    dispute_signals = [
        "dispute", "not valid",
        "not my debt", "paid already",
        "no proof", "validation",
        "verification"
    ]
    score += sum(2 for kw in dispute_signals if kw in text)

    # -------------------------
    # 4. CREDIT REPORT DAMAGE
    # -------------------------
    credit_signals = [
        "credit report", "credit score",
        "reported", "negative",
        "collections account"
    ]
    score += sum(2 for kw in credit_signals if kw in text)

    # -------------------------
    # 5. FDCPA / LEGAL RIGHTS REFERENCES
    # -------------------------
    legal_refs = [
        "fdcpa", "fair debt collection",
        "rights", "violation",
        "illegal"
    ]
    score += sum(2 for kw in legal_refs if kw in text)

    # -------------------------
    # 6. LARGE AMOUNTS
    # -------------------------
    amounts = re.findall(r"\$[\d,]+", text)
    score += min(len(amounts), 4)

    # -------------------------
    # 7. COMPLEXITY / LONG CASE
    # -------------------------
    word_count = len(text.split())
    if word_count > 250:
        score += 2
    elif word_count > 120:
        score += 1

    # -------------------------
    # 8. PERSISTENCE / ONGOING ISSUE
    # -------------------------
    persistence_signals = [
        "months", "years",
        "ongoing", "still",
        "continue", "repeatedly"
    ]
    score += sum(1 for kw in persistence_signals if kw in text)

    # -------------------------
    # FINAL CLASSIFICATION
    # -------------------------
    if score >= 11:
        return "critical"
    elif score >= 5:
        return "high_priority"
    else:
        return "standard"

In [275]:
mask = df["Department"].str.lower() == "debt collection"

df.loc[mask, "priority"] = df.loc[mask, "Consumer complaint narrative"].apply(assign_priority_debt_collection)
df.loc[mask, "priority"].value_counts(normalize=True)

priority
standard         0.412677
high_priority    0.332164
critical         0.255159
Name: proportion, dtype: float64

In [280]:
def assign_priority_card_services(text):
    text = str(text).lower()
    score = 0

    # -------------------------
    # 1. CRITICAL (fraud / unauthorized charges)
    # -------------------------
    fraud_signals = [
        "unauthorized", "fraud", "fraudulent",
        "not authorized", "did not make",
        "identity theft", "hacked", "compromised"
    ]
    score += sum(4 for kw in fraud_signals if kw in text)

    # -------------------------
    # 2. DISPUTE FAILURE (VERY IMPORTANT)
    # -------------------------
    dispute_signals = [
        "dispute", "claim denied",
        "denied", "chargeback",
        "investigation", "refused",
        "not resolved"
    ]
    score += sum(2 for kw in dispute_signals if kw in text)

    # -------------------------
    # 3. LARGE FINANCIAL IMPACT
    # -------------------------
    amounts = re.findall(r"\$[\d,]+", text)
    score += min(len(amounts), 5)

    # -------------------------
    # 4. BILLING / FEE ISSUES
    # -------------------------
    billing_signals = [
        "interest", "late fee",
        "fee", "charged",
        "incorrect charge",
        "overcharged"
    ]
    score += sum(2 for kw in billing_signals if kw in text)

    # -------------------------
    # 5. CARD / ACCOUNT ACCESS ISSUES
    # -------------------------
    access_signals = [
        "card not received", "cannot activate",
        "locked", "restricted",
        "closed", "blocked"
    ]
    score += sum(2 for kw in access_signals if kw in text)

    # -------------------------
    # 6. MERCHANT / DELIVERY ISSUES
    # -------------------------
    merchant_signals = [
        "not received", "never received",
        "package", "delivery",
        "merchant", "product"
    ]
    score += sum(1 for kw in merchant_signals if kw in text)

    # -------------------------
    # 7. COMPLEXITY / LONG CASE
    # -------------------------
    word_count = len(text.split())
    if word_count > 250:
        score += 2
    elif word_count > 120:
        score += 1

    # -------------------------
    # 8. PERSISTENCE / ESCALATION
    # -------------------------
    persistence_signals = [
        "months", "years",
        "multiple times",
        "ongoing", "still"
    ]
    score += sum(1 for kw in persistence_signals if kw in text)

    # -------------------------
    # FINAL CLASSIFICATION
    # -------------------------
    if score >= 12:
        return "critical"
    elif score >= 6:
        return "high_priority"
    else:
        return "standard"

In [281]:
mask = df["Department"].str.lower() == "card services"

df.loc[mask, "priority"] = df.loc[mask, "Consumer complaint narrative"].apply(assign_priority_card_services)
df.loc[mask, "priority"].value_counts(normalize=True)

priority
standard         0.435354
high_priority    0.330494
critical         0.234152
Name: proportion, dtype: float64

In [284]:
def assign_priority_bank_accounts(text):
    text = str(text).lower()
    score = 0

    # -------------------------
    # 1. CRITICAL (fraud / stolen funds)
    # -------------------------
    fraud_signals = [
        "unauthorized", "fraud", "fraudulent",
        "stolen", "identity theft", "not mine",
        "hacked", "compromised"
    ]
    score += sum(3 for kw in fraud_signals if kw in text)

    # -------------------------
    # 2. FUNDS BLOCKED / ACCOUNT LOCKED
    # -------------------------
    access_signals = [
        "frozen", "restricted", "locked",
        "cannot access", "no access",
        "account closed", "closure",
        "terminated relationship"
    ]
    score += sum(3 for kw in access_signals if kw in text)

    # -------------------------
    # 3. MONEY TAKEN / REVERSALS / ERRORS
    # -------------------------
    money_errors = [
        "reversed", "debited", "withdrawn",
        "taken", "overdraft", "fees",
        "charged", "deducted"
    ]
    score += sum(2 for kw in money_errors if kw in text)

    # -------------------------
    # 4. LARGE FINANCIAL IMPACT
    # -------------------------
    amounts = re.findall(r"\$[\d,]+", text)
    score += min(len(amounts), 5)

    # -------------------------
    # 5. DISPUTE / BANK FAILURE
    # -------------------------
    failure_signals = [
        "no response", "not resolved",
        "denied", "claim denied",
        "ignored", "refused",
        "investigation"
    ]
    score += sum(2 for kw in failure_signals if kw in text)

    # -------------------------
    # 6. BASIC BANKING DISRUPTION 
    # -------------------------
    disruption_signals = [
        "deposit hold", "hold",
        "direct deposit", "unable to pay",
        "bills", "rent"
    ]
    score += sum(1 for kw in disruption_signals if kw in text)

    # -------------------------
    # 7. COMPLEXITY / LONG CASE
    # -------------------------
    word_count = len(text.split())
    if word_count > 250:
        score += 2
    elif word_count > 120:
        score += 1

    # -------------------------
    # 8. PERSISTENCE / ESCALATION
    # -------------------------
    persistence_signals = [
        "months", "years",
        "multiple times",
        "still", "ongoing"
    ]
    score += sum(1 for kw in persistence_signals if kw in text)

    # -------------------------
    # FINAL CLASSIFICATION
    # -------------------------
    if score >= 11:
        return "critical"
    elif score >= 6:
        return "high_priority"
    else:
        return "standard"

In [285]:
mask = df["Department"].str.lower() == "bank accounts"

df.loc[mask, "priority"] = df.loc[mask, "Consumer complaint narrative"].apply(assign_priority_bank_accounts)
df.loc[mask, "priority"].value_counts(normalize=True)

priority
standard         0.434313
high_priority    0.315650
critical         0.250037
Name: proportion, dtype: float64

In [288]:
def assign_priority_mortgage(text):
    text = str(text).lower()
    score = 0

    # -------------------------
    # 1. CRITICAL (home loss / legal risk)
    # -------------------------
    critical_signals = [
        "foreclosure", "foreclosed",
        "eviction", "auction", "sheriff sale",
        "bankruptcy", "legal action",
        "default notice", "notice of sale"
    ]
    score += sum(4 for kw in critical_signals if kw in text)

    # -------------------------
    # 2. PAYMENT / ACCOUNT ERRORS 
    # -------------------------
    payment_errors = [
        "misapplied payment", "not applied",
        "incorrect balance", "wrong amount",
        "escrow", "insurance", "taxes",
        "fees", "late fee"
    ]
    score += sum(2 for kw in payment_errors if kw in text)

    # -------------------------
    # 3. CREDIT DAMAGE
    # -------------------------
    credit_signals = [
        "credit report", "credit score",
        "negative reporting", "delinquent"
    ]
    score += sum(2 for kw in credit_signals if kw in text)

    # -------------------------
    # 4. LOAN MODIFICATION / HARDSHIP
    # -------------------------
    hardship_signals = [
        "loan modification", "modification",
        "forbearance", "hardship",
        "covid", "loss mitigation"
    ]
    score += sum(2 for kw in hardship_signals if kw in text)

    # -------------------------
    # 5. SERVICER FAILURE 
    # -------------------------
    servicing_signals = [
        "no response", "not resolved",
        "lost documents", "ignored",
        "run around", "multiple times",
        "submitted documents"
    ]
    score += sum(2 for kw in servicing_signals if kw in text)

    # -------------------------
    # 6. LARGE FINANCIAL IMPACT
    # -------------------------
    amounts = re.findall(r"\$[\d,]+", text)
    score += min(len(amounts), 5)

    # -------------------------
    # 7. LONG TIMELINE / COMPLEXITY
    # -------------------------
    word_count = len(text.split())
    if word_count > 250:
        score += 2
    elif word_count > 120:
        score += 1

    # -------------------------
    # 8. PERSISTENCE / ESCALATION
    # -------------------------
    persistence_signals = [
        "months", "years",
        "still", "ongoing",
        "repeatedly", "many calls"
    ]
    score += sum(1 for kw in persistence_signals if kw in text)

    # -------------------------
    # FINAL CLASSIFICATION
    # -------------------------
    if score >= 11:
        return "critical"
    elif score >= 6:
        return "high_priority"
    else:
        return "standard"

In [289]:
mask = df["Department"].str.lower() == "mortgage"

df.loc[mask, "priority"] = df.loc[mask, "Consumer complaint narrative"].apply(assign_priority_mortgage)
df.loc[mask, "priority"].value_counts(normalize=True)

priority
standard         0.448797
high_priority    0.318981
critical         0.232222
Name: proportion, dtype: float64

In [307]:
def assign_priority_money_transfer(text):
    text = str(text).lower()
    score = 0

    # -------------------------
    # 1. CRITICAL (fraud / unauthorized / funds lost)
    # -------------------------
    critical_signals = [
        "fraud", "scam", "scammed", "unauthorized",
        "compromised", "hacked", "security breach",
        "identity", "stolen", "fake"
    ]
    score += sum(3 for kw in critical_signals if kw in text)

    # -------------------------
    # 2. LOST FUNDS / BLOCKED MONEY
    # -------------------------
    money_loss_signals = [
        "lost", "not received", "missing",
        "never received", "taken out",
        "withdrawn", "chargeback"
    ]
    score += sum(3 for kw in money_loss_signals if kw in text)

    # -------------------------
    # 3. ACCOUNT ACCESS / FREEZE ISSUES
    # -------------------------
    access_signals = [
        "locked", "restricted", "limited",
        "frozen", "deactivated",
        "unable to access", "can not access"
    ]
    score += sum(2 for kw in access_signals if kw in text)

    # -------------------------
    # 4. LARGE MONEY INVOLVEMENT
    # -------------------------
    amounts = re.findall(r"\$[\d,]+", text)
    score += min(len(amounts), 5)

    # -------------------------
    # 5. BANK / PLATFORM FAILURE
    # -------------------------
    failure_signals = [
        "no response", "not resolved", "ignored",
        "refused", "could not help",
        "investigation", "claim denied"
    ]
    score += sum(2 for kw in failure_signals if kw in text)

    # -------------------------
    # 6. URGENCY / DISTRESS
    # -------------------------
    urgency_signals = [
        "urgent", "immediately", "please help",
        "stress", "emotional", "need money",
        "rent", "business harmed"
    ]
    score += sum(1 for kw in urgency_signals if kw in text)

    # -------------------------
    # 7. COMPLEXITY (multi-step fraud cases)
    # -------------------------
    word_count = len(text.split())
    if word_count > 200:
        score += 2
    elif word_count > 100:
        score += 1

    # -------------------------
    # 8. MULTI-TRANSACTION SIGNAL
    # -------------------------
    if len(amounts) > 2:
        score += 1

    # -------------------------
    # FINAL CLASSIFICATION
    # -------------------------
    if score >= 10:
        return "critical"
    elif score >= 5:
        return "high_priority"
    else:
        return "standard"

In [308]:
mask = df["Department"].str.lower() == "money transfer services"

df.loc[mask, "priority"] = df.loc[mask, "Consumer complaint narrative"].apply(assign_priority_money_transfer)
df.loc[mask, "priority"].value_counts(normalize=True)

priority
standard         0.481980
high_priority    0.313544
critical         0.204476
Name: proportion, dtype: float64

In [328]:
def assign_priority_student_loan(text):
    text = str(text).lower()
    score = 0

    # -------------------------
    # 1. CRITICAL (legal + severe harm)
    # -------------------------
    critical_signals = [
        "fraud", "forgery", "identity", "not mine",
        "without consent", "illegal", "violated",
        "lawsuit", "court", "forgiveness ordered",
        "bankruptcy"
    ]
    score += sum(3 for kw in critical_signals if kw in text)

    # -------------------------
    # 2. CREDIT DAMAGE / COLLECTIONS
    # -------------------------
    credit_signals = [
        "credit report", "credit score",
        "collections", "default", "late",
        "negative", "garnish", "wage"
    ]
    score += sum(2 for kw in credit_signals if kw in text)

    # -------------------------
    # 3. SERVICER ERRORS 
    # -------------------------
    servicing_errors = [
        "wrong", "incorrect", "misreported",
        "not applied", "not updated",
        "error", "mistake", "changed terms"
    ]
    score += sum(2 for kw in servicing_errors if kw in text)

    # -------------------------
    # 4. FORGIVENESS / IDR / PSLF ISSUES
    # -------------------------
    forgiveness_signals = [
        "forgiveness", "pslf", "idr",
        "income driven", "deferment",
        "forbearance"
    ]
    score += sum(2 for kw in forgiveness_signals if kw in text)

    # -------------------------
    # 5. HARASSMENT / COLLECTION BEHAVIOR
    # -------------------------
    harassment_signals = [
        "harassing", "calls", "calling",
        "threatening", "voicemail",
        "stress", "litigation"
    ]
    score += sum(1 for kw in harassment_signals if kw in text)

    # -------------------------
    # 6. FINANCIAL STRAIN
    # -------------------------
    distress_signals = [
        "cant afford", "hardship",
        "struggling", "financial",
        "impossible", "desperate"
    ]
    score += sum(1 for kw in distress_signals if kw in text)

    # -------------------------
    # 7. MONEY SIGNALS
    # -------------------------
    amounts = re.findall(r"\$[\d,]+", text)
    score += min(len(amounts), 4)

    # -------------------------
    # 8. COMPLEXITY (long detailed cases)
    # -------------------------
    word_count = len(text.split())
    if word_count > 200:
        score += 2
    elif word_count > 100:
        score += 1

    # -------------------------
    # 9. PERSISTENCE / FAILED RESOLUTION
    # -------------------------
    persistence_signals = [
        "multiple times", "no response",
        "not resolved", "still", "ongoing",
        "hours on the phone"
    ]
    score += sum(2 for kw in persistence_signals if kw in text)

    # -------------------------
    # FINAL CLASSIFICATION
    # -------------------------
    if score >= 10:
        return "critical"
    elif score >= 5:
        return "high_priority"
    else:
        return "standard"

In [329]:
mask = df["Department"].str.lower() == "student loan"

df.loc[mask, "priority"] = df.loc[mask, "Consumer complaint narrative"].apply(assign_priority_student_loan)
df.loc[mask, "priority"].value_counts(normalize=True)

priority
standard         0.381124
high_priority    0.366913
critical         0.251962
Name: proportion, dtype: float64

In [314]:
def assign_priority_consumer_loans(text):
    text = str(text).lower()
    score = 0

    # -------------------------
    # 1. CRITICAL (legal + severe outcomes)
    # -------------------------
    critical_signals = [
        "fraud", "fraudulent", "identity", "forgery",
        "unauthorized", "without consent",
        "repossession", "repo", "stolen",
        "lawsuit", "legal action",
        "violated", "rights", "15 usc"
    ]
    score += sum(3 for kw in critical_signals if kw in text)

    # -------------------------
    # 2. FINANCIAL DAMAGE 
    # -------------------------
    damage_signals = [
        "credit report", "credit score",
        "negative", "collections",
        "balance", "owed", "charged off"
    ]
    score += sum(2 for kw in damage_signals if kw in text)

    # -------------------------
    # 3. LARGE MONEY INVOLVEMENT
    # -------------------------
    amounts = re.findall(r"\$[\d,]+", text)
    if amounts:
        score += min(len(amounts), 4)

    # -------------------------
    # 4. PROCESS FAILURE / MISMANAGEMENT
    # -------------------------
    process_signals = [
        "no response", "not resolved", "refuse",
        "run around", "never", "ignored",
        "incorrect", "wrong", "misreported"
    ]
    score += sum(2 for kw in process_signals if kw in text)

    # -------------------------
    # 5. HARASSMENT / PRESSURE
    # -------------------------
    harassment_signals = [
        "harassing", "calls", "calling",
        "threatening", "voicemail"
    ]
    score += sum(1 for kw in harassment_signals if kw in text)

    # -------------------------
    # 6. CONTRACT / DECEPTION SIGNALS
    # -------------------------
    deception_signals = [
        "misled", "not disclosed", "contract",
        "agreement", "terms", "lied"
    ]
    score += sum(2 for kw in deception_signals if kw in text)

    # -------------------------
    # 7. COMPLEXITY (long structured complaints → higher severity)
    # -------------------------
    word_count = len(text.split())
    if word_count > 200:
        score += 2
    elif word_count > 100:
        score += 1

    # -------------------------
    # 8. MULTI-ISSUE SIGNAL (numbers/dates)
    # -------------------------
    numbers = re.findall(r"\d+", text)
    if len(numbers) > 8:
        score += 1

    # -------------------------
    # FINAL CLASSIFICATION
    # -------------------------
    if score >= 13:
        return "critical"
    elif score >= 7:
        return "high_priority"
    else:
        return "standard"

In [315]:
mask = df["Department"].str.lower() == "consumer loans"

df.loc[mask, "priority"] = df.loc[mask, "Consumer complaint narrative"].apply(assign_priority_consumer_loans)
df.loc[mask, "priority"].value_counts(normalize=True)

priority
standard         0.429844
high_priority    0.321270
critical         0.248887
Name: proportion, dtype: float64

In [318]:
def assign_priority_payday_and_personal_loans(text):
    text = str(text).lower()
    score = 0

    # -------------------------
    # 1. CRITICAL KEYWORDS 
    # -------------------------
    critical_keywords = [
        "fraud", "fraudulent", "scam", "identity theft",
        "unauthorized", "without my consent", "illegal",
        "police", "attorney", "lawsuit", "forgery"
    ]
    score += sum(3 for kw in critical_keywords if kw in text)

    # -------------------------
    # 2. HIGH PRIORITY KEYWORDS
    # -------------------------
    high_keywords = [
        "harassment", "collections", "late fee",
        "incorrect", "wrong", "denied",
        "misleading", "dispute"
    ]
    score += sum(2 for kw in high_keywords if kw in text)

    # -------------------------
    # 3. OUTCOME SEVERITY (actual damage)
    # -------------------------
    severe_outcomes = [
        "credit report", "credit score", "repossession",
        "charged", "taken from my account", "negative report"
    ]
    score += sum(3 for kw in severe_outcomes if kw in text)

    # -------------------------
    # 4. URGENCY / ONGOING SIGNALS
    # -------------------------
    urgency_signals = [
        "still", "continuing", "every day", "constantly",
        "repeatedly", "ongoing"
    ]
    score += sum(1 for kw in urgency_signals if kw in text)

    # -------------------------
    # 5. FAILED RESOLUTION / ESCALATION
    # -------------------------
    escalation_signals = [
        "called multiple times", "no response",
        "they refuse", "not resolved", "ignored"
    ]
    score += sum(2 for kw in escalation_signals if kw in text)

    # -------------------------
    # 6. FINANCIAL MAGNITUDE
    # -------------------------
    amounts = re.findall(r"\$[\d,]+", text)
    score += min(len(amounts), 3)  # cap

    # -------------------------
    # 7. COMPLEXITY (proxy for severity)
    # -------------------------
    word_count = len(text.split())
    if word_count > 150:
        score += 2
    elif word_count > 75:
        score += 1

    # count numbers/dates (timeline/detail signal)
    numbers = re.findall(r"\d+", text)
    if len(numbers) > 5:
        score += 1

    # -------------------------
    # 8. FINANCIAL DISTRESS CONTEXT
    # -------------------------
    distress_keywords = [
        "financial hardship", "cant afford", "desperate",
        "struggling", "hardship"
    ]
    score += sum(1 for kw in distress_keywords if kw in text)

    # -------------------------
    # FINAL CLASSIFICATION
    # -------------------------
    if score >= 9:
        return "critical"
    elif score >= 5:
        return "high_priority"
    else:
        return "standard"

In [319]:
mask = df["Department"].str.lower() == "payday / personal loans"

df.loc[mask, "priority"] = df.loc[mask, "Consumer complaint narrative"].apply(assign_priority_payday_and_personal_loans)
df.loc[mask, "priority"].value_counts(normalize=True)

priority
standard         0.441902
high_priority    0.315194
critical         0.242905
Name: proportion, dtype: float64

In [330]:
def assign_priority_by_department(row):
    dept = str(row["Department"]).lower()
    text = row["Consumer complaint narrative"]

    if "payday" in dept:
        return assign_priority_payday_and_personal_loans(text)
    
    elif "consumer" in dept:
        return assign_priority_consumer_loans(text)
    
    elif "student" in dept:
        return assign_priority_student_loans(text)
    
    elif "money transfer" in dept:
        return assign_priority_money_transfer(text)
    
    elif "mortgage" in dept:
        return assign_priority_mortgage(text)
    
    elif "bank account" in dept:
        return assign_priority_bank_accounts(text)
    
    elif "card" in dept:
        return assign_priority_card_services(text)
    
    elif "debt collection" in dept:
        return assign_priority_debt_collection(text)
    
    elif "credit reporting" in dept:
        return assign_priority_credit_reporting(text)
    
    else:
        return "standard"  # fallback

In [331]:
df["Priority"] = df.apply(assign_priority_by_department, axis=1)

In [332]:
df["Priority"].value_counts(normalize = True)

Priority
standard         0.415716
high_priority    0.331982
critical         0.252301
Name: proportion, dtype: float64

Split data

In [335]:
df_model = df[[
    "Complaint ID",
    "Consumer complaint narrative",
    "Department",
    "Priority"
]].copy()

In [336]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df_model,
    test_size=0.2,
    stratify=df["Department"],
    random_state=42
)

In [337]:
print(train_df["Department"].value_counts(normalize=True))
print(test_df["Department"].value_counts(normalize=True))

Department
Credit reporting           0.582926
Debt collection            0.130451
Card services              0.080420
Bank accounts              0.066205
Mortgage                   0.051714
Money transfer services    0.033293
Student loan               0.021819
Consumer loans             0.020997
Payday / personal loans    0.012174
Name: proportion, dtype: float64
Department
Credit reporting           0.582926
Debt collection            0.130451
Card services              0.080421
Bank accounts              0.066207
Mortgage                   0.051713
Money transfer services    0.033296
Student loan               0.021818
Consumer loans             0.020996
Payday / personal loans    0.012171
Name: proportion, dtype: float64


In [338]:
print(train_df["Priority"].value_counts(normalize=True))
print(test_df["Priority"].value_counts(normalize=True))

Priority
standard         0.415645
high_priority    0.331957
critical         0.252398
Name: proportion, dtype: float64
Priority
standard         0.416001
high_priority    0.332085
critical         0.251915
Name: proportion, dtype: float64


Save modeling split

In [340]:
train_df.to_parquet(
    "../data/processed/train_complaints.parquet",
    index=False,
    compression="snappy"
)

test_df.to_parquet(
    "../data/processed/test_complaints.parquet",
    index=False,
    compression="snappy"
)

Save the cleaned full dataset

In [341]:
df.to_parquet(
    "../data/interim/cfpb_cleaned.parquet",
    index=False,
    compression="snappy"
)